# TP1: Text extraction (PDF + web page + scanned PDF)

This part does three things:

1. Cleanly extract the text from the PDF in this folder (`1706.03762v7.pdf`).
2. Cleanly extract the content of the web article: https://www.elastic.co/search-labs/blog/evaluating-rag-metrics
3. Extract the text from the scanned PDF `matan-90.pdf` ("Handwritten Character Recognition Using Neural Network Architectures") via OCR, since this PDF has no usable text layer (it's an image scan).

Libraries used:
- `pypdf` for the text-based PDF (text extraction, pure Python, no heavy dependency).
- `requests` + `trafilatura` for the web (downloads the page then automatically isolates the article text, stripping menus, footer, ads, etc.).
- `pymupdf` + `pytesseract` + `pillow` for the scanned PDF (rasterizes each page into an image with pymupdf, converts it to a Pillow image, then OCR with Tesseract via pytesseract). Requires Tesseract OCR installed on the machine (system binary, in addition to the pip packages).

In [ ]:
from pathlib import Path

INPUTS_DIR = Path("inputs")

OUTPUTS_DIR = Path("outputs")
OUTPUTS_DIR.mkdir(exist_ok=True)

In [ ]:
# TP1 output folder
TP1_OUTPUT_DIR = OUTPUTS_DIR / "1.clean_scrapping"
TP1_OUTPUT_DIR.mkdir(exist_ok=True)

## Generic cleanup (applied to all 3 sources)

Deliberately source-independent function: it assumes nothing specific to this PDF, this web page, or this scan. It only fixes artifacts generic to any text split into lines (words broken by a hyphen at line end, lines containing only an isolated number — typically a page number, extra whitespace/blank lines).

In [ ]:
import re


def clean_extracted_text(text: str) -> str:
    # rejoin words split by a hyphen at line end ("recog-\nnition" -> "recognition")
    text = re.sub(r"-\n(?=[a-zà-ÿ])", "", text)

    # remove lines containing only an isolated number (typically a page number)
    lines = [line for line in text.splitlines() if not re.fullmatch(r"\d{1,4}", line.strip())]
    text = "\n".join(lines)

    # normalize whitespace and blank lines
    text = "\n".join(line.strip() for line in text.splitlines())
    text = re.sub(r"[ \t]+", " ", text)
    while "\n\n\n" in text:
        text = text.replace("\n\n\n", "\n\n")

    return text.strip()

## 1. PDF extraction

In [111]:
import logging
from pypdf import PdfReader

# silence pypdf's verbose "fontTools is required..." warnings (cosmetic, doesn't affect extraction)
logging.getLogger("pypdf").setLevel(logging.ERROR)

PDF_PATH = INPUTS_DIR / "1.clean_scrapping" / "1706.03762v7.pdf"


def extract_pdf_text(pdf_path: Path) -> str:
    reader = PdfReader(pdf_path)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n\n".join(pages)


pdf_text = clean_extracted_text(extract_pdf_text(PDF_PATH))
print(f"{len(pdf_text)} characters extracted\n")
print(pdf_text[:500])

39431 characters extracted

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


In [ ]:
# Save the extracted text
(TP1_OUTPUT_DIR / "pdf_extract.txt").write_text(pdf_text, encoding="utf-8")

## 2. Web page scraping

In [ ]:
import requests
import trafilatura

URL = "https://www.elastic.co/search-labs/blog/evaluating-rag-metrics"


def extract_web_text(url: str) -> str:
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
    response.raise_for_status()

    # trafilatura automatically isolates the article text
    # (strips nav, footer, cookies, suggested articles, etc.)
    return trafilatura.extract(response.text, include_comments=False, include_tables=True) or ""


web_text = clean_extracted_text(extract_web_text(URL))
print(f"{len(web_text)} characters extracted\n")
print(web_text[:500])

In [ ]:
# Save the extracted text
(TP1_OUTPUT_DIR / "web_extract.txt").write_text(web_text, encoding="utf-8")

## 3. OCR (Optical Character Recognition) extraction of the scanned PDF

`matan-90.pdf` has no text layer (verified: `pypdf` returns an empty string on all 9 pages), it's a pure image scan. So each page is rasterized into an image with `pymupdf`, then OCR is applied with `pytesseract` (requires the Tesseract binary installed on the machine).

In [ ]:
import pymupdf
import pytesseract
from PIL import Image

SCAN_PDF_PATH = INPUTS_DIR / "1.clean_scrapping" / "matan-90.pdf"

pytesseract.pytesseract.tesseract_cmd = r"C:\Users\GDYZ5751\AppData\Local\Programs\Tesseract-OCR\tesseract.exe"


def extract_scanned_pdf_text(pdf_path: Path, dpi: int = 300) -> str:
    doc = pymupdf.open(pdf_path)
    pages_text = []
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        pages_text.append(pytesseract.image_to_string(img))
    return "\n\n".join(pages_text)


ocr_text = clean_extracted_text(extract_scanned_pdf_text(SCAN_PDF_PATH))
print(f"{len(ocr_text)} characters extracted (OCR)\n")
print(ocr_text[:500])

In [ ]:
# Save the extracted text
(TP1_OUTPUT_DIR / "ocr_extract.txt").write_text(ocr_text, encoding="utf-8")

# TP2: Preprocessing (stopword removal, stemming or lemmatization)

Applied to the 3 cleaned texts from TP1. `spaCy` handles tokenization and stopword removal. A global flag, `USE_STEMMING`, picks which normalization runs: `1` uses `nltk`'s `PorterStemmer` (rule-based truncation, e.g. "reli" from "relies"), `0` uses spaCy's lemmatization (dictionary base form, e.g. "rely" from "relies"). Both code paths stay live in the function; the flag just switches between them.

In [ ]:
# TP2 output folder
TP2_OUTPUT_DIR = OUTPUTS_DIR / "2.preprocessing_text"
TP2_OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
import spacy
from nltk.stem import PorterStemmer

# keep "ner" enabled (Named Entity Recognition, used below); disable only "parser" which we don't use
nlp = spacy.load("en_core_web_sm", disable=["parser"])
stemmer = PorterStemmer()

USE_STEMMING = 0  # 1 -> stemming (nltk), 0 -> lemmatization (spaCy)


def preprocess_text(text: str) -> str:
    doc = nlp(text)
    tokens = [token 
              for token in doc 
              if not token.is_stop
              #and not token.is_punct
              ]

    if USE_STEMMING:
        return " ".join(stemmer.stem(token.text.lower()) for token in tokens)
    else:
        return " ".join(token.lemma_.lower() for token in tokens)


pdf_processed = preprocess_text(pdf_text)
web_processed = preprocess_text(web_text)
ocr_processed = preprocess_text(ocr_text)

label = "Stemmed" if USE_STEMMING else "Lemmatized"
print(f"{label} sample:\n", pdf_processed[:300])

In [ ]:
# Save the preprocessed text
suffix = "stemmed" if USE_STEMMING else "lemmatized"

(TP2_OUTPUT_DIR / f"pdf_{suffix}.txt").write_text(pdf_processed, encoding="utf-8")
(TP2_OUTPUT_DIR / f"web_{suffix}.txt").write_text(web_processed, encoding="utf-8")
(TP2_OUTPUT_DIR / f"ocr_{suffix}.txt").write_text(ocr_processed, encoding="utf-8")

## Named entity recognition (NER)

Runs on the cleaned text (`pdf_text`, `web_text`, `ocr_text`), not on the stemmed/lemmatized output above — NER needs the original casing and word boundaries to detect entities correctly. Entities are deduplicated (same text + type kept once) and saved as JSON.

In [ ]:
def extract_entities(text: str) -> list[dict]:
    doc = nlp(text)
    entities = {(ent.text, ent.label_) for ent in doc.ents}
    return [{"text": text, "label": label} for text, label in sorted(entities)]


pdf_entities = extract_entities(pdf_text)
web_entities = extract_entities(web_text)
ocr_entities = extract_entities(ocr_text)

print(f"{len(pdf_entities)} unique entities found in the PDF, e.g.:")
print(pdf_entities[:10])

In [ ]:
import json

# Save the extracted entities
(TP2_OUTPUT_DIR / "pdf_entities.json").write_text(json.dumps(pdf_entities, indent=2, ensure_ascii=False), encoding="utf-8")
(TP2_OUTPUT_DIR / "web_entities.json").write_text(json.dumps(web_entities, indent=2, ensure_ascii=False), encoding="utf-8")
(TP2_OUTPUT_DIR / "ocr_entities.json").write_text(json.dumps(ocr_entities, indent=2, ensure_ascii=False), encoding="utf-8")

# TP3: Movie reviews — sentiment analysis preprocessing

Dataset: `moviereviews.tsv` (2000 labeled movie reviews, `pos`/`neg`). This part covers stats + cleaning; sentiment analysis with `nltk` comes after.

Library used: `pandas` for loading and cleaning the tabular data.

In [ ]:
# TP3 output folder
TP3_OUTPUT_DIR = OUTPUTS_DIR / "3.movie_reviews"
TP3_OUTPUT_DIR.mkdir(exist_ok=True)

## Load & inspect

In [ ]:
import pandas as pd

REVIEWS_PATH = INPUTS_DIR / "3.movie_reviews" / "moviereviews.tsv"

df = pd.read_csv(REVIEWS_PATH, sep="\t")

blank_count = (df["review"].notna() & (df["review"].str.strip() == "")).sum()

print(f"shape: {df.shape}, labels: {df['label'].value_counts().to_dict()}")
print(f"null reviews: {df['review'].isnull().sum()}, blank reviews: {blank_count}")

## Cleaning

Two issues found above: `NaN` reviews, and reviews that are non-null but just whitespace (`.isnull()` alone misses these). Both are dropped. Reviews also contain literal `\r\n` line breaks (preserved by pandas from the quoted multi-line TSV fields) — normalized to spaces.

In [ ]:
# drop missing and blank-only reviews
df_clean = df.dropna(subset=["review"])
df_clean = df_clean[df_clean["review"].str.strip() != ""]

# normalize \r\n line breaks into spaces
df_clean["review"] = df_clean["review"].str.replace(r"\s*\r\n\s*", " ", regex=True) 

df_clean = df_clean.reset_index(drop=True)

print(f"clean shape: {df_clean.shape}, labels: {df_clean['label'].value_counts().to_dict()}")

In [ ]:
# Save the cleaned dataset
df_clean.to_csv(TP3_OUTPUT_DIR / "moviereviews_clean.tsv", sep="\t", index=False)

## Sentiment analysis (VADER)

Pre-built lexicon-based sentiment tool from `nltk` (`SentimentIntensityAnalyzer`) — no training needed. Runs on the raw cleaned reviews (`df_clean["review"]`), not a tokenized/stopword-removed version, since VADER relies on punctuation, capitalization and negation handling that stopword removal would destroy. Each review's compound score is thresholded into `pos`/`neg` and compared to the ground-truth `label` with `sklearn`'s classification report (precision, recall, f1-score, accuracy).

In [ ]:
import nltk

nltk.download("vader_lexicon", quiet=True)

from nltk.sentiment import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer() 

def predict_sentiment(text: str) -> str:
    score = sia.polarity_scores(text)["compound"]
    return "pos" if score >= 0 else "neg"


predicted = df_clean["review"].apply(predict_sentiment)
reference = df_clean["label"]

accuracy = (predicted == reference).mean()
print(f"accuracy: {accuracy:.3f}")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(reference, predicted, labels=["neg", "pos"])

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(reference, predicted)
print(report)

# TP4: Fake news classification — small BERT fine-tuning

Dataset: `fake_news.xlsx` (20800 articles, columns `id`, `title`, `author`, `text`, `label` — `1` = fake, `0` = real, balanced). This part covers stats + cleaning; tokenization and fine-tuning a small BERT come after, step by step.

Libraries used: `pandas` + `openpyxl` (engine to read `.xlsx`).

In [ ]:
# TP4 output folder
TP4_OUTPUT_DIR = OUTPUTS_DIR / "4.fake_news"
TP4_OUTPUT_DIR.mkdir(exist_ok=True)

## Load & inspect

In [ ]:
import pandas as pd

FAKE_NEWS_PATH = INPUTS_DIR / "4.fake_news" / "fake_news.xlsx"

news_df = pd.read_excel(FAKE_NEWS_PATH)

blank_text = (news_df["text"].notna() & (news_df["text"].str.strip() == "")).sum()
duplicate_text = news_df["text"].duplicated().sum()

print(f"shape: {news_df.shape}, labels: {news_df['label'].value_counts().to_dict()}")
print(f"null text: {news_df['text'].isnull().sum()}, blank text: {blank_text}, duplicate text: {duplicate_text}")
news_df.head()

## Cleaning

Same two checks as TP3 (`NaN` text, and non-null but whitespace-only text), plus duplicate articles (same `text` appearing more than once — would leak between train/test splits otherwise; counted after dropping null/blank rows, since `.duplicated()` on the raw data also counts `NaN`s as duplicates of each other, inflating the number). `title` is kept as its own column (missing values filled with `""`) rather than merged into `text`, so it can be fed to BERT's tokenizer as a proper sentence pair (`tokenizer(title, text, ...)`) later. `id` (redundant with the row index) and `author` (risk of the model learning a per-author shortcut instead of real content) are dropped. Articles under 20 words are dropped — inspection showed these are comment/UI fragments (e.g. "Trending", "Seems legit"), not real article bodies, and overwhelmingly labeled fake. Finally, non-English articles are dropped (`langdetect`, checked on the first 1000 characters) — a small minority, but the small BERT used later is an English model.

Each rule's exact removal count is tracked and printed (`removed` dict) instead of hardcoded here, so it can't go stale.

In [ ]:
from IPython.display import display
from langdetect import detect, DetectorFactory

DetectorFactory.seed = 0  # deterministic detection


def is_english(text: str) -> bool:
    try:
        return detect(text[:1000]) == "en"
    except Exception:
        return False


removed = {}

# drop missing text
before = len(news_df)
news_clean = news_df.dropna(subset=["text"])
removed["null text"] = before - len(news_clean)

# drop blank-only text
before = len(news_clean)
news_clean = news_clean[news_clean["text"].str.strip() != ""]
removed["blank text"] = before - len(news_clean)

# drop duplicate articles
before = len(news_clean)
news_clean = news_clean.drop_duplicates(subset=["text"])
removed["duplicate text"] = before - len(news_clean)

# fill missing titles with an empty string; kept as its own column for pair-encoding with BERT's tokenizer later
news_clean["title"] = news_clean["title"].fillna("")

# id is redundant with the row index, author is dropped to avoid the model learning a per-author shortcut instead of real content
news_clean = news_clean[["title", "text", "label"]].reset_index(drop=True)

# drop articles under 20 words in the body (comment/UI fragments, not real article content)
before = len(news_clean)
word_count = news_clean["text"].str.split().str.len()
news_clean = news_clean[word_count >= 20].reset_index(drop=True)
removed["under 20 words"] = before - len(news_clean)

# drop non-English articles (takes a few minutes on the full dataset)
before = len(news_clean)
is_en = news_clean["text"].apply(is_english)
news_clean = news_clean[is_en].reset_index(drop=True)
removed["non-English"] = before - len(news_clean)

removed_summary = pd.Series(removed, name="rows removed")
removed_summary["total"] = removed_summary.sum()
display(removed_summary.to_frame())

print(f"clean shape: {news_clean.shape}, labels: {news_clean['label'].value_counts().to_dict()}")
news_clean.head()

,rows removed
null text,43
blank text,77
duplicate text,301
under 20 words,328
non-English,465
total,1214


clean shape: (19586, 3), labels: {0: 10384, 1: 9202}


,title,text,label
0,House Dem Aide: We Didn’t Even See Comey’s Let...,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Ever get the feeling your life circles the rou...,0
2,Why the Truth Might Get You Fired,"Why the Truth Might Get You Fired October 29, ...",1
3,15 Civilians Killed In Single US Airstrike Hav...,Videos 15 Civilians Killed In Single US Airstr...,1
4,Iranian woman jailed for fictional unpublished...,Print \nAn Iranian woman has been sentenced to...,1


In [ ]:
# Save the cleaned dataset
news_clean.to_csv(TP4_OUTPUT_DIR / "fake_news_clean.csv", index=False)

## TODO — à finir

- Tokenization (`prajjwal1/bert-tiny`, pair-encoding `title`/`text`)
- Chargement du modèle (`BertForSequenceClassification`)
- Split train/validation/test
- Fine-tuning
- Évaluation (classification report + confusion matrix)

# TP5: RAG source scraping

Gathering the source documents for a future RAG: 8 PDFs (`inputs/5.rag_sources/*.pdf`) and 10 web articles. Reuses TP1's generic functions as-is (`extract_pdf_text`, `extract_web_text`) in a plain loop over each source — no per-source custom scraping logic.

Cleaning goes through a new `clean_for_rag` function (not `clean_extracted_text` itself, which TP1-TP4 rely on and which was deliberately kept minimal) that additionally cuts off the bibliography section — never useful for retrieval, and it inflates the chunk count. The cut is detected generically (a standalone `References`/`Bibliography`/`Works Cited` heading), not hardcoded to any specific document, so it works on any input regardless of its content — and simply does nothing on documents that don't have one (blog articles, the OCR'd scan, etc.).

In [137]:
# TP5 input/output folders
TP5_INPUT_DIR = INPUTS_DIR / "5.rag_sources"

TP5_OUTPUT_DIR = OUTPUTS_DIR / "5.rag_sources"
TP5_OUTPUT_DIR.mkdir(exist_ok=True)

In [138]:
def clean_for_rag(text: str) -> str:
    text = clean_extracted_text(text)

    match = re.search(r"^(references|bibliography|works cited)\s*$", text, flags=re.IGNORECASE | re.MULTILINE)
    if match:
        text = text[: match.start()].strip()

    return text

## PDF sources

In [139]:
for pdf_path in sorted(TP5_INPUT_DIR.glob("*.pdf")):
    text = clean_for_rag(extract_pdf_text(pdf_path))
    (TP5_OUTPUT_DIR / f"{pdf_path.stem}.txt").write_text(text, encoding="utf-8")
    print(f"{pdf_path.name}: {len(text)} characters extracted")

Attention_is_all_you_need.pdf: 30008 characters extracted
BERT.pdf: 37210 characters extracted
distilling_knowledge.pdf: 32049 characters extracted
language_understanding_paper_GPT.pdf: 28394 characters extracted
LORA_low_rank_adaptation.pdf: 44199 characters extracted
LUCIE_dataset.pdf: 117541 characters extracted
mixture_of_depth.pdf: 38025 characters extracted
QLORA_finetuning.pdf: 60632 characters extracted


## Web sources

Wrapped in a `try/except`: some sites block simple scrapers (e.g. Medium returns a 403), and one failing source shouldn't stop the rest.

In [140]:
import re

URLS = [
    "https://www.elastic.co/search-labs/blog/evaluating-rag-metrics",
    "https://medium.com/neoxia/deep-dive-in-embeddings-888d4acc1d0a",
    "https://www.evidentlyai.com/llm-guide/llm-as-a-judge#llm-judge-alternatives",
    "https://huggingface.co/blog/manu/colpali",
    "https://weaviate.io/blog/retrieval-evaluation-metrics",
    "https://magazine.sebastianraschka.com/p/the-state-of-llm-reasoning-model-training",
    "https://huggingface.co/blog/clefourrier/llm-evaluation",
    "https://lilianweng.github.io/posts/2024-04-12-diffusion-video/",
    "https://www.geeksforgeeks.org/data-science/what-is-faiss/",
    "https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/",
]


def slugify_url(url: str) -> str:
    path = url.split("://", 1)[-1]
    return re.sub(r"[^a-zA-Z0-9]+", "_", path).strip("_")[:80]


for url in URLS:
    try:
        text = clean_for_rag(extract_web_text(url))
        (TP5_OUTPUT_DIR / f"{slugify_url(url)}.txt").write_text(text, encoding="utf-8")
        print(f"{url}: {len(text)} characters extracted")
    except Exception as e:
        print(f"{url}: FAILED ({e})")

https://www.elastic.co/search-labs/blog/evaluating-rag-metrics: 18061 characters extracted
https://medium.com/neoxia/deep-dive-in-embeddings-888d4acc1d0a: FAILED (403 Client Error: Forbidden for url: https://medium.com/neoxia/deep-dive-in-embeddings-888d4acc1d0a)
https://www.evidentlyai.com/llm-guide/llm-as-a-judge#llm-judge-alternatives: 40842 characters extracted
https://huggingface.co/blog/manu/colpali: 8423 characters extracted
https://weaviate.io/blog/retrieval-evaluation-metrics: 7173 characters extracted
https://magazine.sebastianraschka.com/p/the-state-of-llm-reasoning-model-training: 47636 characters extracted
https://huggingface.co/blog/clefourrier/llm-evaluation: 18673 characters extracted
https://lilianweng.github.io/posts/2024-04-12-diffusion-video/: 21882 characters extracted
https://www.geeksforgeeks.org/data-science/what-is-faiss/: 3582 characters extracted
https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/: 16036 

## Chunking

Splits each scraped document into overlapping chunks for later embedding. Chunk size is measured in **tokens**, not characters or words, since that's the unit embedding models actually work in — counted with `tiktoken`'s `cl100k_base` encoding, a reasonable generic proxy since no embedding model has been picked yet. `CHUNK_SIZE=450` / `CHUNK_OVERLAP=50` tokens (discussed earlier: enough to keep a technical idea intact without diluting the embedding, small overlap so a sentence at a chunk boundary doesn't lose its context). Reads back the `.txt` files just saved above (not the loop variables, which only hold the last document). Each chunk gets an explicit, stable `id` (`"<source>::<i>"`) — needed to keep the embedding vectors aligned with this metadata later, whether that ends up being FAISS (which only stores vectors, so this file becomes the sidecar metadata) or a full vector DB. Saved as one JSON file of `{id, source, text}` records.

In [141]:
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

CHUNK_SIZE = 450
CHUNK_OVERLAP = 50


def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    tokens = encoding.encode(text)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + chunk_size
        chunks.append(encoding.decode(tokens[start:end]))
        start += chunk_size - overlap
    return chunks


txt_paths = sorted(TP5_OUTPUT_DIR.glob("*.txt"))

chunks = []
for txt_path in txt_paths:
    text = txt_path.read_text(encoding="utf-8")
    for i, chunk in enumerate(chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)):
        # id encodes the chunk's position ("<source>::<i>"); split on "::" to recover it if needed
        chunks.append({"id": f"{txt_path.stem}::{i}", "source": txt_path.stem, "text": chunk})

print(f"{len(chunks)} chunks from {len(txt_paths)} documents")

349 chunks from 17 documents


In [142]:
import json

# Save the chunks
(TP5_OUTPUT_DIR / "chunks.json").write_text(json.dumps(chunks, indent=2, ensure_ascii=False), encoding="utf-8")

689109